In [ ]:
import sys
sys.path.append("/project01/ndcms/atownse2/ExponentialMixtureModel")

import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tools import scale_out as so
import emm
import emm.bias as bias

In [ ]:
# Load data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data_tree = emm.get_data(sort_and_index=True, tree=True)
data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = data.numEntries()
data_mean = data.mean(x)
print(f"Data mean: {data_mean}")

In [ ]:
# Set up models
toy_models = [
    emm.f1(x),
    emm.f2(x),
    emm.f3(x),
    emm.f4(x),
]

models = [
    emm.make_model_primitive(emm.ExponentialMixtureModel, 2, data_mean=700, name="ExponentialMixture-2"),
    emm.make_model_primitive(emm.ExponentialMixtureModel, 3, data_mean=700, name="ExponentialMixture-3"),
    emm.make_model_primitive(emm.ExponentialMixtureModel, 4, data_mean=700, name="ExponentialMixture-4"),
]

# Fit toy models to data
for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(data)

In [ ]:
# Configuration
signal_points = [
    (600, 30),
    (700, 32.5),
    (800, 35),
    (900, 37.5),
    (1000, 40),
    (1100, 42.5),
    (1200, 45),
    (1300, 47.5),
    (1400, 50),
    (1500, 52.5),
    (1600, 55),
    (1700, 57.5),
    (1800, 60),
    (1900, 62.5),
    (2000, 65),
]

# Default
n_seeds = 20
n_toys_per_seed = 50

# Test
# n_seeds = 5
# n_toys_per_seed = 1

# Generate seeds
default_seed = 42
np.random.seed(default_seed)
seeds = np.random.randint(0, 10000, size=n_seeds)
print(f"Will be submitting {n_seeds * len(toy_models)} tasks, with {n_toys_per_seed} toys fitting {len(models)} models and {len(signal_points)} signal points.")

In [ ]:
# Run jobs
import os

tasks = []
for seed in seeds:
    for toy_model in toy_models:
        output = bias.signal_fits_filename(toy_model, seed, n_toys_per_seed)
        if os.path.exists(output):
            # print(f"Output {output} already exists, skipping.")
            continue
        _models = models.copy()
        _models.append(type(toy_model))
        tasks.append(so.Task(bias.fit_signal_models, x, toy_model, _models, seed, n_toys_per_seed, n, signal_points))

_ = so.run_tasks(
    tasks,
    use_condor=True,
    condor_job_name="spurious_signal",
    env_wrapper=so.run_in_mamba,
    clear_logs=True
)

In [ ]:
# Load results
all_results = {}
for toy_model in toy_models:
    all_results[toy_model.name] = bias.load_signal_fit_results(toy_model, seeds, n_toys_per_seed)

In [ ]:
# Organize results:
# Each signal point will have a distribution of fitted signal yields for each model
# signal_yields = {toy_model.name: {} for toy_model in toy_models}
all_results_df = []
for toy_model_name, toy_model_results in all_results.items():
    # One model will be the toy model and the others will be the exponential mixtures
    # toy_model_results = all_results[toy_model.name][toy_model.name]

    for seed_uid, signal_point_results in toy_model_results.items():
        for signal_point, fit_results in signal_point_results.items():

            if toy_model_name in fit_results:
                # Save true model
                result = {
                    "toy_model": toy_model_name,
                    "bkg_model": toy_model_name,
                    "seed_uid": seed_uid,
                    "signal_mean": signal_point[0],
                    "signal_width": signal_point[1],
                    "n_sig": fit_results[toy_model_name]["n_sig"],
                    "n_sig_err": fit_results[toy_model_name]["n_sig_err"],
                }
                all_results_df.append(result)

            # Find best exponential mixture
            best_exp_mixture_AIC = None
            best_AIC = np.inf
            best_exp_mixture_BIC = None
            best_BIC = np.inf
            # best_nll = np.inf
            for bkg_model_name, fit_result in fit_results.items():
                if "ExponentialMixture" not in bkg_model_name:
                    continue
                nll = fit_result["bkg_model_nll"]
                n_params = 2 * int(bkg_model_name.split("-")[-1])-1
                AIC = 2 * n_params + 2 * nll
                BIC = n_params * np.log(n) + 2 * nll
                if AIC < best_AIC:
                    best_AIC = AIC
                    best_exp_mixture_AIC = bkg_model_name
                if BIC < best_BIC:
                    best_BIC = BIC
                    best_exp_mixture_BIC = bkg_model_name
            
            result = {
                "toy_model": toy_model_name,
                "bkg_model": "ExponentialMixture (AIC)",
                "seed_uid": seed_uid,
                "signal_mean": signal_point[0],
                "signal_width": signal_point[1],
                "n_sig": fit_results[best_exp_mixture_AIC]["n_sig"],
                "n_sig_err": fit_results[best_exp_mixture_AIC]["n_sig_err"],
            }
            all_results_df.append(result)

            result = {
                "toy_model": toy_model_name,
                "bkg_model": "ExponentialMixture (BIC)",
                "seed_uid": seed_uid,
                "signal_mean": signal_point[0],
                "signal_width": signal_point[1],
                "n_sig": fit_results[best_exp_mixture_BIC]["n_sig"],
                "n_sig_err": fit_results[best_exp_mixture_BIC]["n_sig_err"],
            }
            all_results_df.append(result)

df = pd.DataFrame(all_results_df)

In [ ]:
# Plot results
import matplotlib.pyplot as plt

toy_model_names = df["toy_model"].unique()

fontsize = 18
labelsize = 16
linewidth = 3.5
markersize = 10

colors = {
    "f_1": '#377eb8',
    "f_2": '#ff7f00',
    "f_3": '#4daf4a',
    "f_4": '#f781bf',
    "ExponentialMixture (AIC)": '#984ea3',
    "ExponentialMixture (BIC)": '#a65628',
}
line_styles = ['solid', 'dashed', 'dotted']

fig, axs = plt.subplots(
    len(toy_model_names), 1,
    figsize=(15, 4 * len(toy_model_names)),
    sharex=True,
    gridspec_kw={"hspace": 0}
)
for i, toy_model_name in enumerate(toy_model_names):
    if len(toy_model_names) == 1:
        ax = axs
    else:
        ax = axs[i]

    ax.text(
            0.8,
            0.9,
            f"Truth Model: ${toy_model_name}$",
            transform=ax.transAxes,
            fontsize=fontsize,
            verticalalignment='top',
            horizontalalignment='right',
            color=colors[toy_model_name]
        )

    # toy_model_name = toy_model.name
    toy_model_df = df[df["toy_model"] == toy_model_name]

    # Signal points
    models_to_plot = {
        toy_model_name: (toy_model_name, colors[toy_model_name], line_styles[0]),
        "ExponentialMixture (AIC)": ("Exponential Mixture (AIC)", colors["ExponentialMixture (AIC)"], line_styles[1]),
        "ExponentialMixture (BIC)": ("Exponential Mixture (BIC)", colors["ExponentialMixture (BIC)"], line_styles[2]),
    }

    y_err_true = None

    for bkg_model_name, (label, color, line_style) in models_to_plot.items():
        model_df = toy_model_df[toy_model_df["bkg_model"] == bkg_model_name]
        ms = []
        ys = []
        y_lows = []
        y_highs = []
        for signal_point in signal_points:
            signal_point_df = model_df[
                (model_df["signal_mean"] == signal_point[0]) &
                (model_df["signal_width"] == signal_point[1])
            ]
            
            y = signal_point_df["n_sig"] / signal_point_df["n_sig_err"]
            y_med, y_low, y_high = np.percentile(y, [50, 16, 84])
            ms.append(signal_point[0])
            ys.append(y_med)
            y_lows.append(y_low)
            y_highs.append(y_high)


        if "_" in label:
            label = f"${label}$"
        if bkg_model_name == toy_model_name:
            # label = f"{label} (True Model)"
            label = None

        ax.plot(ms, ys, label=label, marker="o", color=color, linestyle=line_style, linewidth=linewidth, markersize=markersize)
        # print(f"{toy_model_name} - {bkg_model_name}:")
        # print(f"  Signal points: {ms}")
        # print(f"  Spurious signal (S): {ys}")
        if bkg_model_name == "ExponentialMixture (BIC)":
            continue
        ax.fill_between(ms, y_lows, y_highs, alpha=0.3, color=color)
        ax.axhline(0, color="black", linestyle="--", alpha=0.5, linewidth=linewidth)

    if i == 0:
        ax.legend(framealpha=0.3, loc=(0.15, 0.02), fontsize=fontsize, frameon=False)
    ax.set_xlabel("$m_{\gamma\gamma}$ [GeV]", fontsize=fontsize)
    ax.tick_params(axis='x', labelsize=labelsize)
    # ax.set_ylabel("Spurious Signal Yield")
    ax.set_ylabel("$N_s/\sigma_{N_s}$", fontsize=fontsize)
    ax.tick_params(axis='y', labelsize=labelsize)
    ax.set_ylim(-3, 3)
    
